This code is for ai anayze and ai classify

In [0]:
select ai_analyze_sentiment('i am happy') as sentiment;

In [0]:
select ai_classify('my password is leaked')

In [0]:
select ai_parse_document('This is a sample document.') as parsed_document;

In [0]:

SELECT
ai_parse_document(
  content,
  map('version', '2.0')
) AS parsed
FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002449e1-02e3-4ece-b1d4-867992fd44e2.pdf', format => 'binaryFile');


In [0]:
CREATE OR REPLACE TABLE dev_lh.bronze.ai_parsed_documents AS
SELECT
ai_parse_document(
  content,
  map('version', '2.0')
) AS parsed
FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/', format => 'binaryFile');

In [0]:
select * from dev_lh.bronze.ai_parsed_documents;

In [0]:
%python
from pyspark.sql.functions import expr
df = spark.read.format("binaryFile").load("/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002449e1-02e3-4ece-b1d4-867992fd44e2.pdf").withColumn("parsed", expr("ai_parse_document(content)"))
display(df)

In [0]:
SELECT
  path,
  ai_parse_document(content) AS parsed
FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002449e1-02e3-4ece-b1d4-867992fd44e2.pdf', format => 'binaryFile');

In [0]:
SELECT
ai_parse_document(
  content,
  map('version', '2.0')
) AS parsed
FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002a2203-d876-4595-9a90-7f7b368cbe75.pdf', format => 'binaryFile');

In [0]:
%sql
WITH parsed_documents AS (
  SELECT path, ai_parse_document(
    content, Map('imageOutputPath','/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/', 'descriptionElementTypes', '*')
    )AS parsed 
    FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002a2203-d876-4595-9a90-7f7b368cbe75.pdf', format => 'binaryFile')
  ),

In [0]:
%sql
WITH parsed_documents AS (
  SELECT
  path,
  ai_parse_document(content) AS parsed
FROM READ_FILES('/Volumes/dotlas_structured_dataset_from_digitized_restaurant_menu_pdfs_images/samples/menu_documents/menu_pdfs/002449e1-02e3-4ece-b1d4-867992fd44e2.pdf', format => 'binaryFile')
  ),
  parsed_text AS (
    SELECT path, 
    concat_ws(
      '\n\n', 
      transform(
        try_cast(parsed:document:elements AS ARRAY<VARIANT>), 
        element -> try_cast(element:content AS STRING)
      )
    ) AS text
    FROM parsed_documents
    where try_cast(parsed:error_status AS STRING) IS NULL
  )
  select * from parsed_text


In [0]:
--   )
--   SELECT 
--     path, 
--     text, 
--     ai_query(
--       'databricks-meta-llama-34b', 
--       concat(
--         'What is the name of the menu', 
--         text
--         ),
--         returnType => 'STRING'
--       ) AS structured_data
--   from parsed_text
--   WHERE text IS NOT NULL;

-- )

In [0]:
%python
from pyspark.sql.functions import expr
df = spark.read.format("binaryFile").load("/Volumes/dev_lh/bronze/databricks_genai_pdf/LLM_GPT_doc.pdf").withColumn("parsed", expr("ai_parse_document(content)"))
display(df)

In [0]:
%python
# output df into a table  called dev_lh.bronze.ai_parsed_llmdocuments
df.write.format("delta").mode("overwrite").saveAsTable("dev_lh.bronze.ai_parsed_llmdocuments")

In [0]:
select * from dev_lh.bronze.ai_parsed_llmdocuments

In [0]:
%sql
WITH parsed_documents AS (
  SELECT
  path,
  ai_parse_document(content) AS parsed
FROM READ_FILES('/Volumes/dev_lh/bronze/databricks_genai_pdf/LLM_GPT_doc.pdf', format => 'binaryFile')
  ),
  parsed_text AS (
    SELECT path, parsed,
    concat_ws(
      '\n\n', 
      transform(
        try_cast(parsed:document:elements AS ARRAY<VARIANT>), 
        element -> try_cast(element:content AS STRING)
      )
    ) AS text
    FROM parsed_documents
    where try_cast(parsed:error_status AS STRING) IS NULL
  )
  select * from parsed_text


In [0]:
%python
pip install langchain-text-splitters

In [0]:
%skip
%python
from pyspark.sql.functions import expr

ENDPOINT = "databricks-gpt-oss-20b"

prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document, convert the content into clean, readable markdown. Use "==page==" to separate each page. Preserver important structure such as headers, tables and captions. Do not include any JSON or code blocks in the output-just the clean markdown text.

JSON:
'''

transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
             ai_query(
                 '{ENDPOINT}',
                 concat('{prompt_prefix}', CAST(parsed_content AS STRING)),
                 responseFormat => '{{"type": "text"}}
               )
             """)
    )
)
display(transformed_df.select("path", "clean_markdown_text"))


In [0]:
%python
import json
from typing import Any, Optional

def _page_id_from_bbox(bbox: Any) -> Optional[int]:
    """
    bbox can be a list[dict], a dict, or None. Return bbox.page_id if present.
    Kept intentionally simple
    """
    if not bbox:
        return None
    if isinstance(bbox, list) and bbox:
        first = bbox[0] or {}
        return first.get("page_id")
    if isinstance(bbox, dict):
        return bbox.get("page_id")
    return None

def extract_contents_from_json(json_str: str) -> str:
    """
    - Concatenate element 'content' (fallback to 'description' if missing).
    - Insert '== page ==' when page_id changes.
    - Insert an extra newline after elements whose type != 'text'.
    - Return an error string on failure (for easy debugging in DataFrame)
    """
    try:
        doc = json.loads(json_str) if isinstance(json_str, str) else json_str
        if not isinstance(doc, dict):
            return ""
        
        # Support both {"document":{"elements":[...]}} and {"elements":[...]}
        document = doc.get("document", doc)
        elements = document.get("elements", []) if isinstance(document, dict) else []
        if not isinstance(elements, list):
            return ""
        
        out_lines = []
        current_page = None
        
        for el in elements:
            if not isinstance(el, dict):
                continue

            # Page divider on change
            pid = _page_id_from_bbox(el.get("bbox"))
            if pid is not None and current_page is not None and pid != current_page:
                out_lines.append("")
                out_lines.append("== page ==")
                out_lines.append("")
            if pid is not None:
                current_page = pid

            # Content (fallback to description)
            c = el.get("content")
            if not (isinstance(c, str) and c.strip()):
                c = el.get("description")
            if isinstance(c, str):
                out_lines.append(c)

            # Extra newline after non-text elements
            t = (el.get("type") or "").lower()
            if t != "text":
                out_lines.append("")  # produce a blank line after joining

        return "\n".join(out_lines)

    except Exception as e:
        return f"Error: {str(e)}"

# A small factory to create a PySpark UDF that uses the function above
def extract_contents_udf():
    from pyspark.sql.types import StringType
    from pyspark.sql.functions import udf
    @udf(StringType())
    def _udf(json_str):
        try:
            return extract_contents_from_json(json_str)
        except Exception as e:
            return f"Error: {str(e)}"
    return _udf

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import json

extract_contents_udf = extract_contents_udf()

df = spark.read.format("binaryFile").load("/Volumes/dev_lh/bronze/databricks_genai_pdf/LLM_GPT_doc.pdf")
parsed_df = df.withColumn("parsed_content", F.expr("ai_parse_document(content)"))


# Convert VARIANT/struct/map to JSON string
safe_json_col = F.coalesce(
    F.to_json(F.col("parsed_content")),
    F.col("parsed_content").cast("string")
)

# Apply the UDF
plain_text_df = parsed_df.withColumn(
    "plain_text",
    extract_contents_udf(safe_json_col)
)

display(plain_text_df.select("path", "plain_text"))

##Chunk Cleaned Text for Retrieval

```
Now that we have clean, page-separated text, we'll chunk it for retrieval workflows. Chunking helps lanugage maodels and 
vector search systems efficiently process and retrieve relevant information.

we will use LangChain's RecursiveCharacterTextSplitter to split the text by the == page == token, This utility automatically 
handles chunking and overlap, making it easy to prepare text for embedding and search.

Note: Overlap is only introduced when a single input is split into multiple chunks. In this example, each page typically 
forms one chunk with chunk_size = 2000, so some row may not show any overlap. To observer overlapping chunks more clearly,
try reducing the chunk size.

```

In [0]:
%python
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType, BinaryType
import pandas as pd

#set chunking parameters: chunk_size control the max length of each chunk, and chunk_overlap allows for overlapping text between chunks to improve retrieval quality.
CHUNK_SIZE=2000
CHUNK_OVERLAP=200


#Build the text splitter with preferred separators.
# This splitter will break the text at page markers or other natural boundaries, preserving document structure where possible.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["== page ==", "\n\n", "\n", " ", ""]
)

# Define the output schema for the chunked DataFrame
# EAch row will contan the document path and a single text chunk.
schema = StructType([
    StructField("path", StringType(), True),
    StructField("chunk", StringType(), True)
])
       
def split_rows(iterator):
    """
    mapInPandas function: input pdfs with columns [path, plain_text],
    output rows [path, chunk].
    This function splits each document's text into chunks and yields them for DataFrame construction.
    """
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path  = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])
       
# Apply the splitters to plain text DataFrame.
# This step transforms each document into multiple chunked rows for efficient downstream retrieval and embedding.
df_chunks = (
    plain_text_df
    .select("path", "plain_text")
    .mapInPandas(split_rows, schema=schema)
)

# Dsiplay the resulting chunked DataFrame for inspection
display(df_chunks)

In [0]:
%python

# Add a unique, incremental id column before saving
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# save the chunks to a table
df_chunks.write.format("delta").mode("overwrite").saveAsTable("dev_lh.bronze.chunked_llmdocuments")

In [0]:
select * from dev_lh.bronze.chunked_llmdocuments